# IMPORTS

In [1]:
# ==============================
# NLP и текстовая обработка
# ==============================
import re
import string

import nltk
from nltk import FreqDist
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, SnowballStemmer

from razdel import tokenize, sentenize
import pymorphy3  # для русского языка
import spacy

# ==============================
# Работа с данными и векторами
# ==============================
import numpy as np
from numpy.linalg import norm

from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer

# ==============================
# Модели и обучение
# ==============================
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

from sklearn import svm
from sklearn.linear_model import SGDClassifier, LogisticRegression

# ==============================
# Векторные представления слов
# ==============================
import gensim.downloader as api
from gensim.models import Word2Vec

## NLP Practice

### Tokenisation

In [2]:
# @title Default title text
nltk.download('punkt')

[nltk_data] Downloading package punkt to /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
data = "All work and no play makes jack a dull boy, all work and no play"
tokens = word_tokenize(data.lower())  #uses TreeBankWordTokenizer
print(tokens)

['all', 'work', 'and', 'no', 'play', 'makes', 'jack', 'a', 'dull', 'boy', ',', 'all', 'work', 'and', 'no', 'play']


https://www.nltk.org/api/nltk.tokenize.treebank.html#nltk.tokenize.treebank.TreebankWordTokenizer

In [4]:
print(sent_tokenize("I was going home when she rung. It was a surprise."))

['I was going home when she rung.', 'It was a surprise.']


[<img src="https://raw.githubusercontent.com/natasha/natasha-logos/master/natasha.svg">](https://github.com/natasha/natasha)

[Razdel](https://natasha.github.io/razdel/)

In [5]:
text = 'Кружка-термос на 0.5л (50/64 см³, 516;...)'
list(tokenize(text))

[Substring(0, 13, 'Кружка-термос'),
 Substring(14, 16, 'на'),
 Substring(17, 20, '0.5'),
 Substring(20, 21, 'л'),
 Substring(22, 23, '('),
 Substring(23, 28, '50/64'),
 Substring(29, 32, 'см³'),
 Substring(32, 33, ','),
 Substring(34, 37, '516'),
 Substring(37, 38, ';'),
 Substring(38, 41, '...'),
 Substring(41, 42, ')')]

#### RegEx

Everything on regex https://habr.com/ru/post/349860/

In [6]:
word = 'supercalifragilisticexpialidocious'
re.findall('[abc]|up|super', word)

['super', 'c', 'a', 'a', 'c', 'a', 'c']

In [7]:
re.findall('\d{1,3}', 'These are some numbers: 49 and 432312')

['49', '432', '312']

In [8]:
re.sub('[,\.?!]', '', 'How, to? split. text!')

'How to split text'

In [9]:
re.sub('[^A-z]', ' ', 'I 123 can 45 play 67 football').split()

['I', 'can', 'play', 'football']

### Deleting uninformative words

#### N-gramms

In [10]:
unigram = list(nltk.ngrams(tokens, 1))
bigram = list(nltk.ngrams(tokens, 2))
print(unigram[:5])
print(bigram[:5])

[('all',), ('work',), ('and',), ('no',), ('play',)]
[('all', 'work'), ('work', 'and'), ('and', 'no'), ('no', 'play'), ('play', 'makes')]


In [11]:

print('Popular unigramms: ', FreqDist(unigram).most_common(5))
print('Popular bigramms: ', FreqDist(bigram).most_common(5))

Popular unigramms:  [(('all',), 2), (('work',), 2), (('and',), 2), (('no',), 2), (('play',), 2)]
Popular bigramms:  [(('all', 'work'), 2), (('work', 'and'), 2), (('and', 'no'), 2), (('no', 'play'), 2), (('play', 'makes'), 1)]


#### Stopwords

In [12]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
stopWords = set(stopwords.words('english'))
print(stopWords)

{'in', 'but', "won't", 'no', 'such', "she'll", "he's", 'than', 'its', 'few', 'there', "weren't", 'your', 'theirs', "he'll", 'they', 'how', 'she', 'both', 'only', 'ma', 'i', "couldn't", "don't", 'or', 'this', 'doesn', 'not', 'shan', 'when', "hadn't", "haven't", "she's", 'hadn', 'any', 'over', 'an', 'to', 'with', 'what', "i'll", 'myself', "i've", 'which', "mustn't", 'we', 'do', 'here', 'more', "they'll", 'them', 'can', 'who', 'our', 'a', 'below', "shan't", 'her', 're', "you'll", 've', "they're", 'shouldn', 'weren', 'being', 'my', 'down', 'yourselves', 'doing', 'where', 'too', "it'll", "didn't", 'herself', 'were', 'these', 'm', 'ain', 'before', 'because', 'y', 'most', 's', 'why', 'himself', "we've", 'd', 'after', "needn't", 'until', 'am', 'is', 'it', "we'd", 'above', "wasn't", 'did', "you've", 'further', "shouldn't", 'the', "they've", 'yours', 'off', 'yourself', "that'll", 'during', 'don', 'are', 'about', 'at', "it's", 'wouldn', 'up', 'll', 'once', 'mustn', 'very', 'their', "isn't", 'have

In [14]:
print([word for word in tokens if word not in stopWords])

['work', 'play', 'makes', 'jack', 'dull', 'boy', ',', 'work', 'play']


In [15]:
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


#### Stemming and Lemmatization
* ‘Caring’ -> Lemmatization -> ‘Care’
* ‘Caring’ -> Stemming -> ‘Car’

### Stemming
* finding the base form of a word

In [16]:
words = ["game", "gaming", "gamed", "games", "compacted"]

In [17]:
ps = PorterStemmer()
list(map(ps.stem, words))

['game', 'game', 'game', 'game', 'compact']

### Lemmatization
* finding the vocabulary from of a word

In [18]:
raw = """DENNIS: Listen, strange women lying in ponds distributing swords
is no basis for a system of government.  Supreme executive power derives from
a mandate from the masses, not from some farcical aquatic ceremony."""

raw_ru = """Не существует научных доказательств в пользу эффективности НЛП, оно
признано псевдонаукой. Систематические обзоры указывают, что НЛП основано на
устаревших представлениях об устройстве мозга, несовместимо с современной
неврологией и содержит ряд фактических ошибок."""

In [19]:
morph = pymorphy3.MorphAnalyzer()
pymorphy_results = list(map(lambda x: morph.parse(x), raw_ru.split(' ')))
print(' '.join([res[0].normal_form for res in pymorphy_results]))

не существовать научный доказательство в польза эффективность нлп, оно
признать псевдонаукой. систематический обзор указывают, что нлп основать на
устаревший представление о устройство мозга, несовместимый с современной
неврология и содержать ряд фактический ошибок.


In [20]:
# 2
nlp = spacy.load('en_core_web_sm')
spacy_results = nlp(raw)
print(' '.join([token.lemma_ for token in spacy_results]))

DENNIS : listen , strange woman lie in pond distribute sword 
 be no basis for a system of government .   Supreme executive power derive from 
 a mandate from the mass , not from some farcical aquatic ceremony .


### Part-of-Speech

In [21]:
# 1
[(res[0].normal_form, res[0].tag) for res in pymorphy_results[:9]]

[('не', OpencorporaTag('PRCL')),
 ('существовать', OpencorporaTag('VERB,impf,intr sing,3per,pres,indc')),
 ('научный', OpencorporaTag('ADJF,Qual plur,gent')),
 ('доказательство', OpencorporaTag('NOUN,inan,neut plur,gent')),
 ('в', OpencorporaTag('PREP')),
 ('польза', OpencorporaTag('NOUN,inan,femn sing,accs')),
 ('эффективность', OpencorporaTag('NOUN,inan,femn sing,gent')),
 ('нлп,', OpencorporaTag('UNKN')),
 ('оно\nпризнать', OpencorporaTag('PRTS,perf,past,pssv neut,sing'))]

In [22]:
# 2
[(token.lemma_, token.pos_) for token in spacy_results[:7]]

[('DENNIS', 'PROPN'),
 (':', 'PUNCT'),
 ('listen', 'VERB'),
 (',', 'PUNCT'),
 ('strange', 'ADJ'),
 ('woman', 'NOUN'),
 ('lie', 'VERB')]

### Named entities recognition

In [23]:
doc = nlp('Apple is looking at buying U.K. startup for $1 billion')

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Apple 0 5 ORG
U.K. 27 31 GPE
$1 billion 44 54 MONEY


### Classification

#### 20 newsgroups
18000 news, 20 groups

In [24]:
newsgroups_train = fetch_20newsgroups(subset='train')

In [25]:
newsgroups_train.target_names

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [26]:
newsgroups_train.filenames.shape

(11314,)

#### Sample

In [27]:
categories = ['alt.atheism', 'talk.religion.misc',
              'comp.graphics', 'sci.space']
newsgroups_train = fetch_20newsgroups(subset='train',
                                      categories=categories)
newsgroups_train.filenames.shape

(2034,)

In [28]:
print(newsgroups_train.data[0])

From: rych@festival.ed.ac.uk (R Hawkes)
Subject: 3DS: Where did all the texture rules go?
Lines: 21

Hi,

I've noticed that if you only save a model (with all your mapping planes
positioned carefully) to a .3DS file that when you reload it after restarting
3DS, they are given a default position and orientation.  But if you save
to a .PRJ file their positions/orientation are preserved.  Does anyone
know why this information is not stored in the .3DS file?  Nothing is
explicitly said in the manual about saving texture rules in the .PRJ file. 
I'd like to be able to read the texture rule information, does anyone have 
the format for the .PRJ file?

Is the .CEL file format available from somewhere?

Rych

Rycharde Hawkes				email: rych@festival.ed.ac.uk
Virtual Environment Laboratory
Dept. of Psychology			Tel  : +44 31 650 3426
Univ. of Edinburgh			Fax  : +44 31 667 0150



In [29]:
newsgroups_train.target[:10]

array([1, 3, 2, 0, 2, 0, 2, 1, 2, 1])

#### TF-IDF

#### Arguments:
* input : string {‘filename’, ‘file’, ‘content’}
*  lowercase : boolean, default True
*  preprocessor : callable or None (default)
*  tokenizer : callable or None (default)
*  stop_words : string {‘english’}, list, or None (default)
*  ngram_range : tuple (min_n, max_n)
*  max_df : float in range [0.0, 1.0] or int, default=1.0
*  min_df : float in range [0.0, 1.0] or int, default=1
*  max_features : int or None, default=None

In [30]:
# lowercase
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

(2034, 34118)

In [31]:
vectorizer = TfidfVectorizer(lowercase=False)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

(2034, 42307)

In [32]:
vectorizer.get_feature_names_out()[:10]

array(['00', '000', '0000', '00000', '000000', '000005102000', '000021',
       '000062David42', '0000VEC', '0001'], dtype=object)

In [33]:
# min_df, max_df
vectorizer = TfidfVectorizer(min_df=0.8)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

(2034, 9)

In [34]:
vectorizer.get_feature_names_out()

array(['and', 'from', 'in', 'lines', 'of', 'organization', 'subject',
       'the', 'to'], dtype=object)

In [35]:
vectorizer = TfidfVectorizer(min_df=0.01, max_df=0.8)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

(2034, 2391)

In [36]:
# ngram_range
vectorizer = TfidfVectorizer(ngram_range=(1, 3), min_df=0.03, max_df=0.9)
vectors = vectorizer.fit_transform(newsgroups_train.data)
vectors.shape

(2034, 1236)

In [37]:
stopWords = set(stopwords.words('english'))
nltk.download('wordnet')
wnl = nltk.WordNetLemmatizer()


def preproc_nltk(text):
    #text = re.sub(f'[{string.punctuation}]', ' ', text)
    return ' '.join([wnl.lemmatize(word) for word in word_tokenize(text.lower()) if word not in stopWords])


st = "Oh, I think I ve landed Where there are miracles at work,  For the thirst and for the hunger Come the conference of birds"
preproc_nltk(st)

[nltk_data] Downloading package wordnet to /Users/user/nltk_data...


'oh , think landed miracle work , thirst hunger come conference bird'

In [38]:
%%time
vectorizer = TfidfVectorizer(preprocessor=preproc_nltk)
vectors = vectorizer.fit_transform(newsgroups_train.data)

CPU times: user 2.86 s, sys: 7.76 ms, total: 2.86 s
Wall time: 2.88 s


In [39]:
# preproc_spacy
nlp = spacy.load("en_core_web_sm")
texts = newsgroups_train.data.copy()


def preproc_spacy(text):
    spacy_results = nlp(text)
    return ' '.join([token.lemma_ for token in spacy_results if token.lemma_ not in stopWords])


preproc_spacy(st)

'oh , I think I land miracle work ,   thirst hunger come conference bird'

In [40]:
%%time
new_texts = []
for doc in nlp.pipe(texts, batch_size=32, n_process=3, disable=["parser", "ner"]):
    new_texts.append(' '.join([tok.lemma_ for tok in doc if tok.lemma not in stopWords]))
vectorizer = TfidfVectorizer()
vectors = vectorizer.fit_transform(new_texts)

CPU times: user 1.92 s, sys: 269 ms, total: 2.18 s
Wall time: 25.8 s


#### Final Model

In [41]:
vectorizer = TfidfVectorizer(ngram_range=(1, 3), max_df=0.5, max_features=1000)
vectors = vectorizer.fit_transform(new_texts)
vectorizer.get_feature_names_out()[::100]

array(['000', 'attempt', 'child', 'either', 'how', 'little', 'of it',
       'reply', 'technical', 'under'], dtype=object)

#### Cosine similarity

In [42]:
type(vectors)

scipy.sparse._csr.csr_matrix

In [43]:
vector = vectors.todense()[0]
(vector != 0).sum()

np.int64(52)

In [44]:
vector.shape

(1, 1000)

In [45]:
np.mean(list(map(lambda x: (x != 0).sum(), vectors.todense())))

np.float64(89.7866273352999)

In [46]:
dense_vectors = vectors.todense()
dense_vectors.shape

(2034, 1000)

In [47]:
def cosine_sim(v1, v2):
    # v1, v2 (1 x dim)
    return np.array(v1 @ v2.T / norm(v1) / norm(v2))[0][0]

In [48]:
cosine_sim(dense_vectors[0], dense_vectors[0])

np.float64(1.0)

In [49]:
cosines = []
for i in range(10):
    cosines.append(cosine_sim(dense_vectors[0], dense_vectors[i]))

In [50]:
# [1, 3, 2, 0, 2, 0, 2, 1, 2, 1]
cosines

[np.float64(1.0),
 np.float64(0.04340216234273356),
 np.float64(0.005869141279694216),
 np.float64(0.0980722773171457),
 np.float64(0.07085297166768148),
 np.float64(0.06711110069455452),
 np.float64(0.02673898595765374),
 np.float64(0.22858574223732228),
 np.float64(0.03200549805311869),
 np.float64(0.06908546765867311)]

#### Training a model on features

In [51]:





X_train, X_test, y_train, y_test = train_test_split(dense_vectors, newsgroups_train.target, test_size=0.2,
                                                    random_state=0)
y_train.shape, y_test.shape

((1627,), (407,))

In [52]:
X_train = np.asarray(X_train)
y_train = np.asarray(y_train)
X_test = np.asarray(X_test)
y_test = np.asarray(y_test)

In [53]:
sgd = SGDClassifier()
sgd.fit(X_train, y_train)
accuracy_score(y_test, sgd.predict(X_test))

0.9090909090909091

### Embeddings

In [54]:

embeddings_pretrained = api.load('glove-twitter-25')

[==================================================] 100.0% 104.8/104.8MB downloaded


In [55]:


proc_words = [preproc_nltk(text).split() for text in newsgroups_train.data]
embeddings_trained = Word2Vec(proc_words,  # data for model to train on
                              vector_size=100,  # embedding vector size
                              min_count=3,  # consider words that occured at least 5 times
                              window=3).wv

In [56]:
def vectorize_sum(comment, embeddings):
    """
    implement a function that converts preprocessed comment to a sum of token vectors
    """
    embedding_dim = embeddings.vectors.shape[1]
    features = np.zeros([embedding_dim], dtype='float32')

    for word in preproc_nltk(comment).split():
        if word in embeddings:
            features += embeddings[f'{word}']

    return features

In [57]:
len(embeddings_trained.index_to_key)

13553

In [58]:
X_wv = np.stack([vectorize_sum(text, embeddings_pretrained) for text in newsgroups_train.data])
X_train_wv, X_test_wv, y_train, y_test = train_test_split(X_wv, newsgroups_train.target, test_size=0.2, random_state=0)
X_train_wv.shape, X_test_wv.shape

((1627, 25), (407, 25))

In [59]:



clf = LogisticRegression(max_iter=5000)
wv_model = clf.fit(X_train_wv, y_train)
accuracy_score(y_test, wv_model.predict(X_test_wv))

0.6953316953316954

In [60]:
X_wv = np.stack([vectorize_sum(text, embeddings_trained) for text in newsgroups_train.data])
X_train_wv, X_test_wv, y_train, y_test = train_test_split(X_wv, newsgroups_train.target, test_size=0.2, random_state=0)
X_train_wv.shape, X_test_wv.shape

((1627, 100), (407, 100))

In [61]:



clf = LogisticRegression(max_iter=10000)
wv_model = clf.fit(X_train_wv, y_train)
accuracy_score(y_test, wv_model.predict(X_test_wv))

/Users/user/PycharmProjects/MLC/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 10000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=10000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


0.8353808353808354